# Greek Medical Report Anonymization

This notebook anonymizes Greek medical reports directly inside Google Colab.

**How to use**

1. Run the setup cell.
2. Run the model-loading cell.
3. Run the upload-and-process cell.
4. Review the full anonymized report(s) inside Colab.
5. Download the generated `.zip` file.


In [ ]:
#@title Step 1 - Setup
print('Setting up the notebook environment...')

%cd /content
!rm -rf greek-medical-anonymization
!git clone https://github.com/VanessaLislevand/greek-medical-anonymization.git
%cd /content/greek-medical-anonymization
%pip install -q -e ".[ml]" gdown

print('Setup completed.')


In [ ]:
#@title Step 2 - Settings and Model Loading

REPORT_TYPE = "Report with template and free text" #@param ["Report with template and free text", "Free-text-only report"]
MASK_TOKEN = "[REDACTED]" #@param {type:"string"}

MODEL_SHARE_LINK = "https://drive.google.com/file/d/1RIHFqp5Xke7t5JtMuXJBhoR_gqEVUoO_/view?usp=share_link"

import os
import shutil
import sys
from pathlib import Path
import zipfile

from IPython.display import Markdown, display
import gdown

repo_root = Path('/content/greek-medical-anonymization')
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from greek_med_anonymizer.config import AppConfig, ModelConfig, RuleConfig
from greek_med_anonymizer.pipeline import AnonymizationPipeline

PROCESSING_MODE = {
    'Report with template and free text': 'auto',
    'Free-text-only report': 'free_text_only',
}[REPORT_TYPE]


model_download_dir = Path('/content/model_download')
model_extract_root = Path('/content/model_artifacts')
model_zip_path = model_download_dir / 'xlmr_phi_final.zip'

shutil.rmtree(model_download_dir, ignore_errors=True)
shutil.rmtree(model_extract_root, ignore_errors=True)
model_download_dir.mkdir(parents=True, exist_ok=True)
model_extract_root.mkdir(parents=True, exist_ok=True)

print('Downloading model archive...')
gdown.download(MODEL_SHARE_LINK, str(model_zip_path), quiet=False, fuzzy=True)

print('Extracting model archive...')
with zipfile.ZipFile(model_zip_path, 'r') as archive:
    archive.extractall(model_extract_root)

required_files = {'config.json', 'tokenizer.json', 'tokenizer_config.json'}
weight_filenames = {'model.safetensors', 'pytorch_model.bin'}

candidate_dirs = []
for path in [model_extract_root] + [p for p in model_extract_root.rglob('*') if p.is_dir()]:
    files_in_dir = {child.name for child in path.iterdir() if child.is_file()}
    if required_files.issubset(files_in_dir) and files_in_dir.intersection(weight_filenames):
        candidate_dirs.append(path)

if not candidate_dirs:
    raise FileNotFoundError('The extracted model folder was not found.')

model_extract_dir = min(candidate_dirs, key=lambda p: len(p.parts))

config = AppConfig(
    input_glob='*.docx',
    output_suffix='.anon.txt',
    mask_token=MASK_TOKEN,
    processing_mode=PROCESSING_MODE,
    rules=RuleConfig(phones=True, patient_ids=True),
    model=ModelConfig(
        enabled=True,
        model_dir=str(model_extract_dir),
        labels_to_mask=['PHI'],
        aggregation_strategy='simple',
    ),
)

pipeline = AnonymizationPipeline(config)

display(Markdown('### Pipeline ready'))
print(f'Report type: {REPORT_TYPE}')
print(f'Mask token: {MASK_TOKEN}')
print(f'Model folder: {model_extract_dir}')
print('Model files found:')
for path in sorted(model_extract_dir.iterdir()):
    if path.is_file():
        print(f'- {path.name}')


In [ ]:
#@title Step 3 - Upload, Process, and Download

import sys
from io import BytesIO
import json
from pathlib import Path
import tempfile
import zipfile

from IPython.display import Markdown, display
import ipywidgets as widgets
from google.colab import files

repo_root = Path('/content/greek-medical-anonymization')
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from greek_med_anonymizer.io_utils import read_input_text


def entity_payload(entities):
    return [
        {
            'start': entity.start,
            'end': entity.end,
            'label': entity.label,
            'text': entity.text,
            'source': entity.source,
        }
        for entity in entities
    ]


def build_output_name(filename: str) -> str:
    path = Path(filename)
    stem = path.stem
    if path.parent != Path('.'):
        safe_parent = '__'.join(path.parent.parts)
        return f'{safe_parent}__{stem}.anon.txt'
    return f'{stem}.anon.txt'


def anonymize_payload(filename: str, payload: bytes):
    suffix = Path(filename).suffix or '.txt'
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as temp_file:
        temp_file.write(payload)
        temp_path = Path(temp_file.name)

    try:
        text = read_input_text(temp_path)
        result = pipeline.anonymize_text(text)
    finally:
        temp_path.unlink(missing_ok=True)

    return {
        'filename': filename,
        'output_name': build_output_name(filename),
        'anonymized_text': result.anonymized_text,
        'metadata': entity_payload(result.entities),
    }


print('Upload one or more .docx/.txt files, or a .zip archive containing reports.')
print('To test another file or folder, simply run this cell again.')
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No files were uploaded.')

output_zip_path = Path('/content/anonymized_outputs.zip')
full_outputs = []
processed_count = 0

with zipfile.ZipFile(output_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as output_archive:
    for filename, payload in uploaded.items():
        suffix = Path(filename).suffix.lower()

        if suffix == '.zip':
            with zipfile.ZipFile(BytesIO(payload)) as source_archive:
                for member_name in source_archive.namelist():
                    if member_name.endswith('/'):
                        continue
                    if Path(member_name).suffix.lower() not in {'.docx', '.txt'}:
                        continue

                    summary = anonymize_payload(member_name, source_archive.read(member_name))
                    output_archive.writestr(summary['output_name'], summary['anonymized_text'])
                    output_archive.writestr(
                        summary['output_name'] + '.json',
                        json.dumps(summary['metadata'], ensure_ascii=False, indent=2),
                    )
                    full_outputs.append(summary)
                    processed_count += 1
        elif suffix in {'.docx', '.txt'}:
            summary = anonymize_payload(filename, payload)
            output_archive.writestr(summary['output_name'], summary['anonymized_text'])
            output_archive.writestr(
                summary['output_name'] + '.json',
                json.dumps(summary['metadata'], ensure_ascii=False, indent=2),
            )
            full_outputs.append(summary)
            processed_count += 1
        else:
            print(f'Skipping unsupported file: {filename}')

display(Markdown(f'### Anonymization completed for {processed_count} file(s)'))
print(f'Output archive: {output_zip_path}')
print('You can rerun this cell at any time with different files.')

for item in full_outputs:
        display(Markdown(f"## {item['filename']}"))
        display(Markdown(f"**Detected entities:** {len(item['metadata'])}"))
        print(item['anonymized_text'])
        print('\n' + '=' * 100 + '\n')

download_button = widgets.Button(description='Download output ZIP', button_style='success')
download_status = widgets.Output()

def on_download_click(_):
    with download_status:
        download_status.clear_output()
        print('Preparing download...')
    files.download(str(output_zip_path))

download_button.on_click(on_download_click)
display(download_button)
display(download_status)
